<a href="https://colab.research.google.com/github/SamyraLB/Portfolio-de-Projetos-Dados-ADA-e-BI/blob/main/01_gerador_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Definir semente para reprodutibilidade
np.random.seed(42)
random.seed(42)

# --- 1. GERAR DIM_CLIENTES ---
num_clientes = 950
portes = ['Pequeno', 'Médio', 'Grande']
prob_portes = [0.5, 0.35, 0.15]
segmentos = ['Tecnologia', 'Varejo', 'Construção Civil', 'Saúde', 'Logística', 'Financeiro']

clientes = []
data_inicio_base = datetime(2022, 1, 1)

for i in range(1, num_clientes + 1):
    id_cli = f"CLI-{i:03d}"
    nome = f"Empresa {chr(65 + (i % 26))}{i}"
    porte = np.random.choice(portes, p=prob_portes)
    segmento = random.choice(segmentos)

    # Data de início do relacionamento entre 2022 e meados de 2024
    dias_add = random.randint(0, 800)
    data_inicio = data_inicio_base + timedelta(days=dias_add)

    # 15% de chance de o cliente ter sido cancelado
    status = np.random.choice(['Ativo', 'Cancelado'], p=[0.85, 0.15])

    clientes.append({
        'id_cliente': id_cli,
        'nome_cliente': nome,
        'porte_cliente': porte,
        'segmento_atuacao': segmento,
        'data_inicio_relacionamento': data_inicio.strftime('%Y-%m-%d'),
        'status_cliente': status
    })

df_clientes = pd.DataFrame(clientes)

# --- 2. GERAR FATO_FATURAMENTO ---
#--m
faturas = []
areas = ['Trabalhista', 'Cível', 'Tributário']
modalidades = ['Partido Fixo', 'Êxito']

# Período de emissão: 01/01/2024 a 30/06/2026
data_inicio_fat = datetime(2024, 1, 1)
data_fim_fat = datetime(2026, 6, 30)

id_fatura = 10001

for idx, cliente in df_clientes.iterrows():
    # Definição do volume de faturas por porte
    if cliente['porte_cliente'] == 'Grande':
        qtd_faturas = random.randint(20, 35)
    elif cliente['porte_cliente'] == 'Médio':
        qtd_faturas = random.randint(5, 20)
    else:
        qtd_faturas = random.randint(3, 5)

    for _ in range(qtd_faturas):
        # Data de emissão aleatória dentro do período
        dias_aleatorios = random.randint(0, (data_fim_fat - data_inicio_fat).days)
        dt_emissao = data_inicio_fat + timedelta(days=dias_aleatorios)

        # Garante que a fatura seja posterior ao início do relacionamento
        dt_cli = datetime.strptime(cliente['data_inicio_relacionamento'], '%Y-%m-%d')
        if dt_emissao < dt_cli:
            dt_emissao = dt_cli + timedelta(days=random.randint(1, 60))

        dt_vencimento = dt_emissao + timedelta(days=30)

        area = random.choice(areas)
        modalidade = np.random.choice(modalidades, p=[0.7, 0.3])

        # Lógica de Valores baseada na Área e Modalidade
        if modalidade == 'Partido Fixo':
            valor_base = random.choice([3000, 5000, 8000, 12000, 20000])
        else: # Êxito (valores maiores e mais variáveis)
            valor_base = random.choice([15000, 25000, 45000, 80000])

        if cliente['porte_cliente'] == 'Grande':
            valor_faturado = valor_base * 1.8
        else:
            valor_faturado = valor_base

        # Regra de Inadimplência / Pagamento (Injetando padrão de negócio)
        # Trabalhista e Êxito têm maior taxa de atraso/inadimplência
        rand_status = random.random()

        if dt_vencimento > datetime(2026, 6, 1):
            # Faturas muito recentes podem estar pendentes/em aberto
            status_cobranca = 'Em Aberto'
            dt_pagamento = None
            valor_pago = 0.0
        elif rand_status < 0.82: # 82% PAGO EM DIA OU PEQUENO ATRASO
            status_cobranca = 'Pago'
            dias_atraso = random.choice([0, 0, 0, 2, 5, 10, 15])
            dt_pagamento = (dt_vencimento + timedelta(days=dias_atraso)).strftime('%Y-%m-%d')
            valor_pago = valor_faturado
        elif rand_status < 0.93: # 11% PAGO COM MUITO ATRASO
            status_cobranca = 'Pago com Atraso'
            dias_atraso = random.randint(31, 90)
            dt_pagamento = (dt_vencimento + timedelta(days=dias_atraso)).strftime('%Y-%m-%d')
            valor_pago = valor_faturado
        else: # 7% INADIMPLENTE / PERDA (WRITE-OFF)
            status_cobranca = 'Inadimplente'
            dt_pagamento = None
            valor_pago = 0.0

        faturas.append({
            'id_faturamento': f"FAT-{id_fatura}",
            'id_cliente': cliente['id_cliente'],
            'area_direito': area,
            'modalidade_contrato': modalidade,
            'data_emissao': dt_emissao.strftime('%Y-%m-%d'),
            'data_vencimento': dt_vencimento.strftime('%Y-%m-%d'),
            'data_pagamento': dt_pagamento,
            'valor_faturado': round(valor_faturado, 2),
            'valor_pago': round(valor_pago, 2),
            'status_cobranca': status_cobranca
        })
        id_fatura += 1

df_faturamento = pd.DataFrame(faturas)

# Exportar para CSV
df_clientes.to_csv('dim_clientes.csv', index=False)
df_faturamento.to_csv('fato_faturamento.csv', index=False)

print("Arquivos 'dim_clientes.csv' e 'fato_faturamento.csv' gerados com sucesso!")

Arquivos 'dim_clientes.csv' e 'fato_faturamento.csv' gerados com sucesso!
